In [1]:
import pandas as pd
import numpy as np

# ==============================
# 1. Load processed dataset
# ==============================

df = pd.read_csv(
    "data/market_data_processed.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Original Shape:", df.shape)


# ==============================
# 2. Crash Definition
# ==============================

LOOKAHEAD = 10
CRASH_THRESHOLD = -0.07

prices = df["SP500"]


# ==============================
# 3. Future Minimum Price
# ==============================

# For each day:
# Find minimum S&P 500 price over NEXT 10 trading days.
#
# shift(-1) ensures today's price is NOT included.

future_min = (
    prices
    .shift(-1)
    .iloc[::-1]
    .rolling(
        window=LOOKAHEAD,
        min_periods=LOOKAHEAD
    )
    .min()
    .iloc[::-1]
)

df["Future_Min_SP500"] = future_min


# ==============================
# 4. Future Drawdown
# ==============================

df["Future_Drawdown_10D"] = (
    df["Future_Min_SP500"] / df["SP500"]
) - 1


# ==============================
# 5. Create Crash Label
# ==============================

df["Crash_Risk"] = (
    df["Future_Drawdown_10D"]
    <= CRASH_THRESHOLD
).astype(int)


# ==============================
# 6. Remove Unknown Future Rows
# ==============================

# Last 10 rows don't have a complete future window

df = df.dropna(
    subset=["Future_Drawdown_10D"]
)


# ==============================
# 7. Validation
# ==============================

print("\nFinal Shape:")
print(df.shape)

print("\nCrash Distribution:")
print(df["Crash_Risk"].value_counts())

print("\nCrash Percentage:")
print(
    df["Crash_Risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ==============================
# 8. Worst Future Drawdowns
# ==============================

print("\nTop 20 Highest Crash-Risk Dates:")

print(
    df[
        [
            "SP500",
            "Future_Drawdown_10D",
            "Crash_Risk"
        ]
    ]
    .sort_values(
        "Future_Drawdown_10D"
    )
    .head(20)
)


# ==============================
# 9. Check Known Crisis Periods
# ==============================

periods = {
    "Dot-Com / 9-11":
        ("2001-01-01", "2002-12-31"),

    "Financial Crisis":
        ("2008-01-01", "2009-06-30"),

    "COVID Crash":
        ("2020-01-01", "2020-06-30")
}

for name, (start, end) in periods.items():

    subset = df.loc[start:end]

    print(f"\n{name}")

    print(
        "Crash-risk days:",
        subset["Crash_Risk"].sum()
    )

    if subset["Crash_Risk"].sum() > 0:

        print(
            subset[
                subset["Crash_Risk"] == 1
            ][
                [
                    "SP500",
                    "Future_Drawdown_10D"
                ]
            ]
            .head(10)
        )


# ==============================
# 10. Save
# ==============================

df.to_csv(
    "data/market_data_labeled.csv"
)

print(
    "\nSaved: data/market_data_labeled.csv"
)

Original Shape: (6462, 26)

Final Shape:
(6452, 29)

Crash Distribution:
Crash_Risk
0    6185
1     267
Name: count, dtype: int64

Crash Percentage:
Crash_Risk
0    95.86
1     4.14
Name: proportion, dtype: float64

Top 20 Highest Crash-Risk Dates:
                  SP500  Future_Drawdown_10D  Crash_Risk
Date                                                    
2008-09-26  1213.270020            -0.258846           1
2008-09-25  1209.180054            -0.247490           1
2020-03-04  3130.120117            -0.237687           1
2008-09-30  1166.359985            -0.229037           1
2020-03-02  3090.229980            -0.227847           1
2008-10-01  1161.060059            -0.225518           1
2020-03-06  2972.370117            -0.224552           1
2020-03-10  2882.229980            -0.223726           1
2020-03-05  3023.939941            -0.210920           1
2020-03-03  3003.370117            -0.205516           1
2008-10-02  1114.280029            -0.193004           1
2008-11-07